This is the yt clipper download tool! To get started, click the "Run All" button in the header bar.

In [1]:
#@title Setup exporter

import os
import re
import glob
import shutil
import subprocess
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
except ImportError:
    !pip install ipywidgets -q
    import ipywidgets as widgets

from google.colab import files

WORKDIR = "/content/yt_clipper_work"
os.makedirs(WORKDIR, exist_ok=True)

display(HTML("""
<div style="
    background:#111;
    color:white;
    padding:22px;
    border-radius:12px;
    font-family:Arial, sans-serif;
    margin-bottom:16px;
">
    <h2 style="margin-top:0;">✂️ YT Clipper Exporter</h2>
    <p style="font-size:16px;color:#ddd;">
        Paste the <b>yt-dlp command</b> copied from YT Clipper, then click <b>Run Export</b>.
    </p>
    <p style="font-size:13px;color:#aaa;">
        This notebook automatically clears old video files before each run so Colab storage does not fill up.
    </p>
</div>
"""))

print("Installing/updating yt-dlp...")
subprocess.run(["python", "-m", "pip", "install", "-U", "yt-dlp", "-q"], check=True)

print("Checking ffmpeg...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg", "-qq"], check=True)

print("Setup complete.")
print("Working folder:", WORKDIR)

Installing/updating yt-dlp...
Checking ffmpeg...
Setup complete.
Working folder: /content/yt_clipper_work


In [3]:
#@title Paste command and export

import os
import re
import glob
import html
import time
import shutil
import threading
import subprocess
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import files

WORKDIR = "/content/yt_clipper_work"
os.makedirs(WORKDIR, exist_ok=True)

last_output_file = {
    "path": None
}

# -----------------------------
# File cleanup helpers
# -----------------------------

def clean_workdir():
    """
    Deletes old exported and temporary media files from the working folder.
    This prevents Colab storage from filling up.
    """
    patterns = [
        "*.mp4",
        "*.webm",
        "*.mkv",
        "*.m4a",
        "*.part",
        "*.ytdl",
        "part_*",
        "inputs.txt",
        "*.txt",
        "run_export.sh",
    ]

    deleted = 0

    for pattern in patterns:
        for path in glob.glob(os.path.join(WORKDIR, pattern)):
            try:
                if os.path.isfile(path) or os.path.islink(path):
                    os.remove(path)
                    deleted += 1
                elif os.path.isdir(path):
                    shutil.rmtree(path)
                    deleted += 1
            except Exception as e:
                print(f"Could not delete {path}: {e}")

    return deleted


def cleanup_temp_files():
    """
    Keeps final YT_*.mp4 files but removes temporary part files and inputs.txt.
    """
    patterns = [
        "part_*",
        "inputs.txt",
        "*.part",
        "*.ytdl",
        "run_export.sh",
    ]

    for pattern in patterns:
        for path in glob.glob(os.path.join(WORKDIR, pattern)):
            try:
                if os.path.isfile(path) or os.path.islink(path):
                    os.remove(path)
                elif os.path.isdir(path):
                    shutil.rmtree(path)
            except Exception as e:
                print(f"Could not clean temp file {path}: {e}")


# -----------------------------
# Tool checks
# -----------------------------

def ensure_tools():
    """
    Makes sure yt-dlp and ffmpeg are available.
    Cell 1 should already install them, but this makes the exporter more reliable.
    """
    print("Checking tools...")

    try:
        subprocess.run(
            ["yt-dlp", "--version"],
            check=True,
            capture_output=True,
            text=True
        )
        print("yt-dlp is installed.")
    except Exception:
        print("Installing/updating yt-dlp...")
        subprocess.run(
            ["python", "-m", "pip", "install", "-U", "yt-dlp", "-q"],
            check=True
        )

    try:
        subprocess.run(
            ["ffmpeg", "-version"],
            check=True,
            capture_output=True,
            text=True
        )
        print("ffmpeg is installed.")
    except Exception:
        print("Installing ffmpeg...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "ffmpeg", "-qq"], check=True)


# -----------------------------
# Command helpers
# -----------------------------

def normalize_command(command):
    """
    Cleans up common copy/paste issues:
    - Converts &amp;&amp; back to &&
    - Normalizes line endings
    - Removes trailing spaces after line-continuation backslashes
    """
    cmd = command.strip()

    # Fix HTML-escaped command text, e.g. &amp;&amp;
    cmd = html.unescape(cmd)

    # Normalize line endings
    cmd = cmd.replace("\r\n", "\n").replace("\r", "\n")

    # Bash line-continuation backslash must be the last character on the line
    cmd = re.sub(r"\\[ \t]+\n", "\\\n", cmd)

    return cmd.strip()


def looks_like_safe_yt_clipper_command(command):
    """
    Basic guardrails so users do not accidentally run unrelated shell commands.

    This is not a perfect security sandbox, but it helps catch obvious mistakes.
    """
    cmd = command.strip()

    if not cmd:
        return False, "Please paste a command first."

    if "yt-dlp" not in cmd:
        return False, "This does not look like a yt-dlp command."

    if "--download-sections" not in cmd:
        return False, "This command does not include any --download-sections entries."

    if "youtube.com/watch?v=" not in cmd and "youtu.be/" not in cmd:
        return False, "This command does not appear to contain a YouTube video URL."

    blocked_patterns = [
        r"\brm\s+-rf\s+/",
        r"\bsudo\b",
        r"\bchmod\b",
        r"\bchown\b",
        r"\bdd\b",
        r"\bmkfs\b",
        r":\(\)\s*\{",
    ]

    for pattern in blocked_patterns:
        if re.search(pattern, cmd):
            return False, f"Blocked potentially unsafe command pattern: {pattern}"

    return True, "OK"


def find_latest_output():
    """
    Finds the final MP4 generated by the export.
    Prefers YT_*.mp4, then falls back to newest .mp4.
    """
    outputs = sorted(
        glob.glob(os.path.join(WORKDIR, "YT_*.mp4")),
        key=os.path.getmtime,
        reverse=True
    )

    if outputs:
        return outputs[0]

    mp4s = sorted(
        glob.glob(os.path.join(WORKDIR, "*.mp4")),
        key=os.path.getmtime,
        reverse=True
    )

    if mp4s:
        return mp4s[0]

    return None


# -----------------------------
# UI widgets
# -----------------------------

command_box = widgets.Textarea(
    value="",
    placeholder='Paste the yt-dlp command copied from YT Clipper here...',
    description="Command:",
    layout=widgets.Layout(width="100%", height="260px")
)

run_button = widgets.Button(
    description="Run Export",
    button_style="success",
    layout=widgets.Layout(width="180px", height="44px")
)

download_button = widgets.Button(
    description="Download MP4",
    button_style="info",
    disabled=True,
    layout=widgets.Layout(width="180px", height="44px")
)

clear_button = widgets.Button(
    description="Clear Files",
    button_style="warning",
    layout=widgets.Layout(width="180px", height="44px")
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Progress:",
    bar_style="info",
    layout=widgets.Layout(width="100%")
)

status_label = widgets.HTML(
    value="<b>Status:</b> Waiting for command..."
)

output_area = widgets.Output()


# -----------------------------
# Progress helpers
# -----------------------------

def set_progress(value, status, style="info"):
    progress_bar.value = value
    progress_bar.bar_style = style
    status_label.value = f"<b>Status:</b> {status}"


def mark_failed(message="Export failed."):
    progress_bar.bar_style = "danger"
    status_label.value = f"<b>Status:</b> ❌ {message}"


def mark_success(message="Export complete!"):
    progress_bar.value = 100
    progress_bar.bar_style = "success"
    status_label.value = f"<b>Status:</b> ✅ {message}"


# -----------------------------
# Live bash runner with animated progress
# -----------------------------

def run_live_bash_with_progress(script_path):
    """
    Runs the export shell script and streams output live.
    While the command is running, the progress bar animates between 40% and 80%.
    """
    stop_flag = {"stop": False}

    def animate_progress():
        value = 40
        direction = 1

        while not stop_flag["stop"]:
            progress_bar.value = value

            value += direction

            if value >= 80:
                direction = -1
            elif value <= 40:
                direction = 1

            time.sleep(0.5)

    thread = threading.Thread(target=animate_progress)
    thread.daemon = True
    thread.start()

    try:
        process = subprocess.Popen(
            ["/bin/bash", script_path],
            cwd=WORKDIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="")

        process.wait()
        return process.returncode

    finally:
        stop_flag["stop"] = True
        thread.join(timeout=1)


# -----------------------------
# Button callbacks
# -----------------------------

def on_run_clicked(button):
    with output_area:
        clear_output()

        download_button.disabled = True
        run_button.disabled = True
        clear_button.disabled = True

        last_output_file["path"] = None

        set_progress(0, "Starting export...", "info")

        raw_command = command_box.value
        command = normalize_command(raw_command)

        ok, message = looks_like_safe_yt_clipper_command(command)

        if not ok:
            mark_failed("Cannot run command.")
            print("Cannot run command.")
            print(message)

            run_button.disabled = False
            clear_button.disabled = False
            return

        try:
            print("Preparing export...")
            print()

            set_progress(10, "Checking yt-dlp and ffmpeg...", "info")
            ensure_tools()
            print()

            set_progress(20, "Cleaning old files to save Colab storage...", "info")
            print("Cleaning old files so Colab storage does not fill up...")
            deleted = clean_workdir()
            print(f"Deleted {deleted} old file(s).")
            print()

            set_progress(30, "Preparing export script...", "info")

            script_path = os.path.join(WORKDIR, "run_export.sh")

            script_contents = f"""#!/bin/bash
set -o pipefail
cd "{WORKDIR}"

echo "Working directory:"
pwd
echo

echo "yt-dlp version:"
yt-dlp --version
echo

echo "Files before export:"
ls -lh
echo

echo "Running export..."
echo

{command}

EXIT_CODE=$?

echo
echo "Command exit code: $EXIT_CODE"
echo

echo "Files after export:"
ls -lh
echo

exit $EXIT_CODE
"""

            with open(script_path, "w") as f:
                f.write(script_contents)

            os.chmod(script_path, 0o755)

            set_progress(
                40,
                "Downloading and processing clips. This may take a few minutes...",
                "warning"
            )

            print("Running export command...")
            print("Live output will appear below.")
            print()

            return_code = run_live_bash_with_progress(script_path)

            if return_code != 0:
                mark_failed("Export failed.")
                print()
                print("Export failed.")
                print("Scroll up and look for the first yt-dlp or ffmpeg error message.")
                print()
                print("Common causes:")
                print("- YouTube blocked Colab and requires sign-in")
                print("- The video is unavailable from Colab")
                print("- The pasted command was modified accidentally")
                print("- ffmpeg concat failed")

                run_button.disabled = False
                clear_button.disabled = False
                return

            set_progress(85, "Cleaning temporary files...", "info")
            cleanup_temp_files()

            set_progress(92, "Finding final MP4...", "info")
            output_path = find_latest_output()

            if not output_path:
                mark_failed("No MP4 output file was found.")
                print("Export finished, but no MP4 output file was found.")
                print("Files in working folder:")
                print(os.listdir(WORKDIR))

                run_button.disabled = False
                clear_button.disabled = False
                return

            size_mb = os.path.getsize(output_path) / (1024 * 1024)

            last_output_file["path"] = output_path
            download_button.disabled = False

            mark_success("Export complete. Ready to download.")

            print()
            print("Export complete!")
            print("File:", os.path.basename(output_path))
            print(f"Size: {size_mb:.2f} MB")
            print()
            print("Click Download MP4 below.")

        except Exception as e:
            mark_failed("Something went wrong.")
            print("Something went wrong:")
            print(e)

        finally:
            run_button.disabled = False
            clear_button.disabled = False


def on_download_clicked(button):
    path = last_output_file.get("path")

    if path and os.path.exists(path):
        files.download(path)
    else:
        with output_area:
            print("No MP4 is ready yet. Click Run Export first.")


def on_clear_clicked(button):
    with output_area:
        clear_output()

        set_progress(0, "Clearing files...", "warning")

        deleted = clean_workdir()

        download_button.disabled = True
        last_output_file["path"] = None

        set_progress(0, "Files cleared. Ready for a new command.", "info")

        print(f"Cleared {deleted} file(s) from the working folder.")


run_button.on_click(on_run_clicked)
download_button.on_click(on_download_clicked)
clear_button.on_click(on_clear_clicked)


# -----------------------------
# Display UI
# -----------------------------

display(HTML("""
<div style="
    background:#1b1b1b;
    color:white;
    padding:16px;
    border-radius:10px;
    font-family:Arial, sans-serif;
    margin-bottom:12px;
">
    <h3 style="margin-top:0;">Instructions</h3>
    <ol style="line-height:1.7;">
        <li>Go back to YT Clipper.</li>
        <li>Click <b>Copy Command</b>.</li>
        <li>Paste the command into the box below.</li>
        <li>Click <b>Run Export</b>.</li>
        <li>When finished, click <b>Download MP4</b>.</li>
    </ol>
    <p style="color:#aaa;font-size:13px;margin-bottom:0;">
        Tip: The progress bar moves while the export is running. Long videos or many clips may take quite a while.
    </p>
</div>
"""))

display(command_box)
display(widgets.HBox([run_button, download_button, clear_button]))
display(status_label)
display(progress_bar)
display(output_area)


Textarea(value='', description='Command:', layout=Layout(height='260px', width='100%'), placeholder='Paste the…

HTML(value='<b>Status:</b> Waiting for command...')

IntProgress(value=0, bar_style='info', description='Progress:', layout=Layout(width='100%'))

Output()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>